# P2-A7 — The Same Agent, in LangGraph

In P2-A6 you hand-wrote the agent loop. Now you'll build the **same thing** in **LangGraph** — a widely-used framework for building production agents. The goal isn't to learn magic; it's to *recognise your own loop* inside the framework, so LangGraph feels like a tool, not a black box.

Two parts:
- **Task 1:** the prebuilt agent — your whole `run_agent` in ~2 lines.
- **Task 2:** the custom graph — the *same loop* drawn as nodes + edges, so you see what the prebuilt one does inside.

New idea: a **graph** of nodes (steps) connected by edges (transitions). The agent loop is a graph: an *agent* node and a *tools* node, with an edge that loops back while there are still tool calls.

In [15]:
# --- Setup (given) ---
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
import grandalf

load_dotenv()
llm = ChatOpenAI(model='gpt-4o-mini')   # reads OPENAI_API_KEY from .env

# Same order system as P2-A6, but tools are now LangChain @tool functions.
# Note: the @tool decorator auto-generates the JSON schema from your signature +
# docstring — no hand-written schema like P2-A5/A6. That's the framework earning its keep.
ORDERS = {
    'A123': {'status': 'shipped',    'eta_days': 2, 'placed': '2026-06-20'},
    'B456': {'status': 'processing', 'eta_days': 5, 'placed': '2026-06-27'},
}

@tool
def list_orders() -> list:
    """List all of the customer's orders with the date each was placed."""
    return [{'order_id': k, 'placed': v['placed']} for k, v in ORDERS.items()]

@tool
def get_order_status(order_id: str) -> dict:
    """Get the status and ETA of one order by its ID."""
    return ORDERS.get(order_id, {'status': 'not_found'})

@tool
def cancel_order(order_id: str) -> dict:
    """Cancel one order by its ID."""
    if order_id in ORDERS:
        ORDERS[order_id]['status'] = 'cancelled'
        return {'order_id': order_id, 'result': 'cancelled'}
    return {'order_id': order_id, 'result': 'not_found'}

TOOLS = [list_orders, get_order_status, cancel_order]
print('tools:', [t.name for t in TOOLS])

tools: ['list_orders', 'get_order_status', 'cancel_order']


## Task 1 — The prebuilt agent (your whole loop, in 2 lines)
Use LangChain's prebuilt agent and run the same multi-step question from P2-A6.
<br>Hint:
```python
from langchain.agents import create_agent
agent = create_agent(llm, TOOLS)
result = agent.invoke({'messages': [{'role': 'user', 'content': "What's the status of my most recent order?"}]})
print(result['messages'][-1].content)
```
Compare to your P2-A6 `run_agent`: the framework collapsed the entire loop into one call.

In [16]:
# TODO: build the prebuilt agent and invoke it on the 'most recent order' question

from langchain.agents import create_agent

agent = create_agent(llm, TOOLS)
result = agent.invoke({'messages': [
    {'role': 'user', 'content': "What's the status of my most recent order?"}
]})
print(result['messages'][-1].content)


Your most recent order (ID: B456), placed on June 27, 2026, is currently in processing status. The estimated time of arrival (ETA) is 5 days.


## Task 2 — Build the custom graph (see your loop as nodes + edges)
Now build the loop yourself with a `StateGraph`. This is *exactly* your P2-A6 loop, expressed as a graph:
- an **`agent` node** = the model call (your `client.chat.completions.create`)
- a **`tools` node** = `ToolNode(TOOLS)` (your dispatch + run)
- a **conditional edge** = `tools_condition` (your `if msg.tool_calls`): if the agent asked for tools → go to `tools`; else → `END`
- `tools` always loops **back to `agent`** (your `while`)
<br>Fill the TODOs:
```python
bound = llm.bind_tools(TOOLS)
def agent_node(state):
    # TODO: invoke `bound` on state['messages'], return {'messages': [the_response]}
graph = StateGraph(MessagesState)
# TODO: add_node('agent', agent_node); add_node('tools', ToolNode(TOOLS))
# TODO: add_edge(START, 'agent'); add_conditional_edges('agent', tools_condition); add_edge('tools', 'agent')
app = graph.compile()
```

In [17]:
# TODO: build the StateGraph (agent node + tools node + edges) and compile it to `app`
bound = llm.bind_tools(TOOLS)

def agent_node(state):
    response = bound.invoke(state['messages'])   # the model call (P2-A6's create(...))
    return {'messages': [response]}              # appended to MessagesState

graph = StateGraph(MessagesState)
graph.add_node('agent', agent_node)
graph.add_node('tools', ToolNode(TOOLS))         # dispatch + run (P2-A6's TOOL_REGISTRY)

graph.add_edge(START, 'agent')
graph.add_conditional_edges('agent', tools_condition)  # if tool_calls -> 'tools', else END
graph.add_edge('tools', 'agent')                 # loop back (P2-A6's while)

app = graph.compile()
print('graph compiled')


graph compiled


## Task 3 — Run your graph (and see its shape)
Invoke your compiled `app` on *"Please cancel my most recent order."*, print the final answer, and verify `ORDERS['B456']` is now `cancelled`.
<br>Hint: invoke the same way as Task 1 — `app.invoke({'messages': [...]})`.

In [ ]:
# invoke `app` on the cancel task; print final answer + verify ORDERS
result = app.invoke({'messages': [
    {'role': 'user', 'content': 'Please cancel my most recent order.'}
]})
print(result['messages'][-1].content)
print()
print('B456 status:', ORDERS['B456']['status'])   # -> 'cancelled'

Your most recent order (ID: B456) has been successfully canceled.

B456 status: cancelled


## Task 4 — Explain (in your own words)
1. Map each part of your P2-A6 `run_agent` to a piece of the LangGraph graph (the `while`, the model call, the tool dispatch, the `if tool_calls` check, the stop condition).
2. Given you can hand-roll the loop, what do you actually *gain* by using LangGraph? Name two things a framework gives you that you'd otherwise have to build yourself (think: state, persistence, streaming, human-in-the-loop, observability).

> _your answer here_
1. The mapping is one-to-one with P2-A6's run_agent:

the while loop → the cyclic edge tools → agent (the graph re-enters the agent node instead of my for step in range(max_steps));
the model call (client.chat.completions.create) → the agent node (bound.invoke(state['messages']));
the tool dispatch (TOOL_REGISTRY[name](**args) + appending the role='tool' message) → the tools node (ToolNode(TOOLS), which runs each call and appends results);
the if msg.tool_calls check → the conditional edge tools_condition;
the stop condition (return text when there are no tool calls) → that same conditional edge routing to END. And my hand-managed messages list → LangGraph's MessagesState, which accumulates messages automatically.
2. Even though I can hand-roll the loop, the framework gives me production plumbing for free: (a) persistence / state management — a checkpointer can save MessagesState after every step, so a conversation can be paused, resumed, or recovered after a crash, and threads keep their own memory (I'd otherwise have to build a store and reload logic myself). (b) human-in-the-loop + streaming/observability — LangGraph lets me interrupt the graph before a node (e.g. pause before cancel_order for approval — the exact guardrail I flagged in P2-A6), stream tokens/steps as they happen, and trace every node transition. Other things it hands me: retries/error handling, parallel/branching control flow, and easy swapping of nodes — all of which I'd otherwise write and maintain by hand.